In [1]:
import pickle
import pandas
import os
from Tools.DatasetTools.DatasetOperator import Dataset, DatasetTester

In [2]:
target_case = 'EF_nmhcp'

In [3]:
ModelName = 'Kernel Ridge'

In [4]:
import logging
Logger = logging.getLogger()
Logger.setLevel(logging.INFO)
logging.basicConfig(format='%(message)s')

In [5]:
namefile = ModelName.replace(' ', '')
suffix = f"no_hcp_bcc_fcc_{namefile}" #'CV_restart_folds_inloop''CV_restart_folds_inloop

In [6]:
dataset = 'Fe-Mo'

In [7]:
DS = Dataset('Fe-Mo', target_name=target_case, remove_phases_query='Phase != "bcc" and Phase != "fcc" and Phase !="hcp"')

In [8]:
feature_concat_resul_loc = os.path.join(dataset, 'results', f'concatenation_results_{target_case}_{suffix}.pkl')  
if os.path.exists(feature_concat_resul_loc):
    with open(feature_concat_resul_loc, 'rb') as pkl:
        savedFCresults = pickle.load(pkl)
else:
    savedFCresults = {}

In [9]:
savedFCresults[(ModelName, 'atomic no CNAV')]

[                                     train1     test1  \
 MagpieData mean GSmagmom           0.144313  0.146840   
 Structure                          0.127471  0.138823   
 MagpieData maximum Row             0.125671  0.142002   
 MagpieData minimum AtomicWeight    0.123107  0.141536   
 MagpieData mode NsValence          0.115461  0.139977   
 MagpieData avg_dev Number          0.105383  0.141100   
 Mag                                0.097627  0.146766   
 MagpieData mode Electronegativity  0.093689  0.148566   
 
                                                                               params  \
 MagpieData mean GSmagmom           {'regressor__alpha': 1.0, 'regressor__coef0': ...   
 Structure                          {'regressor__alpha': 1.0, 'regressor__coef0': ...   
 MagpieData maximum Row             {'regressor__alpha': 1.0, 'regressor__coef0': ...   
 MagpieData minimum AtomicWeight    {'regressor__alpha': 1.0, 'regressor__coef0': ...   
 MagpieData mode NsValence     

In [10]:
cleanfcresults = {}
for combi, fcresults in savedFCresults.items():
    ncurves = len(fcresults)
    if ModelName not in combi:
        Logger.info(f'{combi} is not {ModelName}')
        continue
    if ncurves < 1:
        Logger.info(f'{combi} has no curves')
        continue
    Logger.info(f'combi: {combi}, curves: {ncurves}')
    cleanfcresults[combi] = []
    for i, curve in enumerate(fcresults):
        if 'random' in curve.index:
            random_in = curve.index.get_loc('random')
        else:
            random_in = len(curve.index)
        cleancurve = curve.iloc[:,:random_in]
        nfeats = cleancurve.shape[0]
        if nfeats > 0:
            cleanfcresults[combi].append(curve)
#            Logger.info(f'curve {i}, nselected = {curve.shape}, random in: {random_in}, features clean: {nfeats}')

combi: ('Kernel Ridge', 'atomic'), curves: 3
combi: ('Kernel Ridge', 'dataset'), curves: 3
combi: ('Kernel Ridge', 'Canonical ACE'), curves: 12
combi: ('Kernel Ridge', 'Canonical BOP'), curves: 13
combi: ('Kernel Ridge', 'ACE no CNAV'), curves: 10
combi: ('Kernel Ridge', 'ACE'), curves: 12
combi: ('Kernel Ridge', '0.7dProjections 0.5OS BOP'), curves: 12
combi: ('Kernel Ridge', '0.7spProjections 0.5OS BOP'), curves: 13
combi: ('Kernel Ridge', 'atomic no CNAV'), curves: 10
combi: ('Kernel Ridge', 'dataset no CNAV'), curves: 10
combi: ('Kernel Ridge', '0.7dProjections 0.5OS BOP no CNAV'), curves: 10
combi: ('Kernel Ridge', '0.7spProjections 0.5OS BOP no CNAV'), curves: 10
combi: ('Kernel Ridge', 'SOAP_specific_small'), curves: 10
combi: ('Kernel Ridge', 'SOAP_specific_small no CNAV'), curves: 10
combi: ('Kernel Ridge', 'Canonical ACE no CNAV'), curves: 10
combi: ('Kernel Ridge', 'Canonical BOP no CNAV'), curves: 67
combi: ('Kernel Ridge', 'SOAP_canonicalW_small'), curves: 10
combi: ('Kern

test by learning

In [11]:
from Tools.DatasetTools.MLConveniences import filter_features
import joblib
dataset='Fe-Mo'

In [12]:
models = [ModelName]

In [13]:
modelnames=[model.replace(' ','') for model in models]

In [14]:
voting_regressor_files = {modelname: os.path.join(dataset, 'results', f'voting_regressor_{modelname}.pkl')
                         for modelname in modelnames}

In [15]:
voting_regressor_files

{'KernelRidge': 'Fe-Mo/results/voting_regressor_KernelRidge.pkl'}

In [16]:
optimal_regressors = {}
for model in modelnames:
    optimal_regressors.update(joblib.load(voting_regressor_files[model]))

In [17]:
redbold = "\x1b[31;1m"
reset = "\x1b[0m"

In [18]:
problematic_estimators = {}
for ( model, featurename ), votingregressor in optimal_regressors.items():
    estimators = votingregressor.estimators
    Logger.info(f'model:{model}, featurename: {featurename}, nestimators :{len(estimators)}' )
    X = DS.Features[featurename]
    problematic_estimators[(model, featurename)]=[]
    for iestimator, estimator in estimators:
        learning_curve = estimator.named_steps.feature_selection.kw_args['learning_curve']
        nfeatures = learning_curve.shape
        transformed_X =estimator.named_steps.feature_selection.transform(X)
        if transformed_X.shape[1] < 1:
            Logger.warning(f'{redbold}index{iestimator}, transformed curve length is ZERO (0){reset}')
            problematic_estimators[(model, featurename)].append(iestimator)
            continue
        Logger.info(f'index:{iestimator}, transformer curve length: {nfeatures}, nfeatures stranformed: {transformed_X.shape}')


model:Kernel Ridge, featurename: atomic, nestimators :3
index:0, transformer curve length: (8, 5), nfeatures stranformed: (262, 1)
index:1, transformer curve length: (8, 5), nfeatures stranformed: (262, 2)
index:2, transformer curve length: (8, 5), nfeatures stranformed: (262, 1)
model:Kernel Ridge, featurename: dataset, nestimators :3
index:0, transformer curve length: (21, 5), nfeatures stranformed: (262, 8)
index:1, transformer curve length: (8, 5), nfeatures stranformed: (262, 4)
index:2, transformer curve length: (17, 5), nfeatures stranformed: (262, 7)
model:Kernel Ridge, featurename: Canonical ACE, nestimators :12
index:0, transformer curve length: (202, 5), nfeatures stranformed: (262, 72)
index:1, transformer curve length: (199, 5), nfeatures stranformed: (262, 74)
index:2, transformer curve length: (132, 5), nfeatures stranformed: (262, 98)
index:3, transformer curve length: (95, 5), nfeatures stranformed: (262, 4)
index:4, transformer curve length: (143, 5), nfeatures stranf

In [19]:
problematic = savedFCresults[(ModelName, 'atomic no CNAV')][4]

In [20]:
optimal_regressors[(ModelName, 'atomic no CNAV')]

VotingRegressor(estimators=[('0',
                             Pipeline(steps=[['feature_selection',
                                              FunctionTransformer(func=<function filter_features at 0x7fee8d8b5c10>,
                                                                  kw_args={'learning_curve':                                      train1     test1  \
MagpieData mean GSmagmom           0.144313  0.146840   
Structure                          0.127471  0.138823   
MagpieData maximum Row             0.125671  0.142002   
MagpieData minimum AtomicWeight    0.123107  0.141536   
MagpieData mode NsValence          0.115461...
MagpieData minimum AtomicWeight  0.147183  0.144261  
Structure                        0.134221  0.131155  
MagpieData maximum MeltingT      0.137975  0.125466  
Mag                              0.147515  0.142368  
MagpieData avg_dev Row           0.146598  0.139062  
MagpieData mode GSmagmom         0.147410  0.138883  
MagpieData mode NUnfilled        0.150883  0.138520  ,
                                                                           'remove_structure': True})],
                                             ('scaler', StandardScaler()),
                                             ('regressor',
                                              KernelRidge(alpha=1.0, degree=2,
                                                          kernel='polynomial'))]))])

In [21]:
problematic_estimator = optimal_regressors[(ModelName, 'atomic no CNAV')].estimators.pop(4)

In [22]:
problematic_estimator[1].named_steps.feature_selection.transform(X)

KeyError: "None of [Index(['MagpieData mean AtomicWeight'], dtype='object')] are in the [columns]"

In [23]:
DS.Features['atomic no CNAV'][problematic_estimator[1].named_steps.feature_selection.kw_args['learning_curve'].index]

,MagpieData mean AtomicWeight,Structure,MagpieData maximum AtomicWeight,MagpieData minimum MeltingT,MagpieData mode AtomicWeight,MagpieData mode NValence,Mag
Fe_pv4Mo_sv20.C36-ABBBB.FM,89.274167,3,95.960,1811.0,95.960,6.0,1
Fe_pv15Mo_sv38.R-AAAABBBBBBB.NM,84.606698,4,95.960,1811.0,95.960,6.0,0
Fe_pv2Mo_sv11.mu-BBABB.FM,89.788462,9,95.960,1811.0,95.960,6.0,1
Fe_pv8Mo_sv22.sigma-BBBAB.NM,85.262667,10,95.960,1811.0,95.960,6.0,0
Fe_pv2Mo_sv11.mu-BBBBA.NM,89.788462,9,95.960,1811.0,95.960,6.0,0
...,...,...,...,...,...,...,...
Fe_pv8Mo_sv16.C36-BAABB.NM,82.588333,3,95.960,1811.0,95.960,6.0,0
Fe_pv30.sigma.FM,55.845000,10,55.845,1811.0,55.845,8.0,1
Fe_pv6.C15.FM,55.845000,2,55.845,1811.0,55.845,8.0,1
Mo_sv8.A15.NM,95.960000,0,95.960,2896.0,95.960,6.0,0


In [24]:
DS.Features['atomic no CNAV'][problematic.index]

,MagpieData mean AtomicWeight,Structure,MagpieData maximum AtomicWeight,MagpieData minimum MeltingT,MagpieData mode AtomicWeight,MagpieData mode NValence,Mag,random,MagpieData avg_dev AtomicWeight
Fe_pv4Mo_sv20.C36-ABBBB.FM,89.274167,3,95.960,1811.0,95.960,6.0,1,0.461308,11.143056
Fe_pv15Mo_sv38.R-AAAABBBBBBB.NM,84.606698,4,95.960,1811.0,95.960,6.0,0,0.465851,16.280206
Fe_pv2Mo_sv11.mu-BBABB.FM,89.788462,9,95.960,1811.0,95.960,6.0,1,0.374388,10.444142
Fe_pv8Mo_sv22.sigma-BBBAB.NM,85.262667,10,95.960,1811.0,95.960,6.0,0,0.767237,15.689422
Fe_pv2Mo_sv11.mu-BBBBA.NM,89.788462,9,95.960,1811.0,95.960,6.0,0,0.698486,10.444142
...,...,...,...,...,...,...,...,...,...
Fe_pv8Mo_sv16.C36-BAABB.NM,82.588333,3,95.960,1811.0,95.960,6.0,0,0.883622,17.828889
Fe_pv30.sigma.FM,55.845000,10,55.845,1811.0,55.845,8.0,1,0.859332,0.000000
Fe_pv6.C15.FM,55.845000,2,55.845,1811.0,55.845,8.0,1,0.329511,0.000000
Mo_sv8.A15.NM,95.960000,0,95.960,2896.0,95.960,6.0,0,0.249109,0.000000


In [25]:
problematic_estimators

{('Kernel Ridge', 'atomic'): [],
 ('Kernel Ridge', 'dataset'): [],
 ('Kernel Ridge', 'Canonical ACE'): [],
 ('Kernel Ridge', 'Canonical BOP'): [],
 ('Kernel Ridge', 'ACE no CNAV'): [],
 ('Kernel Ridge', 'ACE'): [],
 ('Kernel Ridge', '0.7dProjections 0.5OS BOP'): [],
 ('Kernel Ridge', '0.7spProjections 0.5OS BOP'): [],
 ('Kernel Ridge', 'atomic no CNAV'): [],
 ('Kernel Ridge', 'dataset no CNAV'): [],
 ('Kernel Ridge', '0.7dProjections 0.5OS BOP no CNAV'): [],
 ('Kernel Ridge', '0.7spProjections 0.5OS BOP no CNAV'): [],
 ('Kernel Ridge', 'SOAP_specific_small'): [],
 ('Kernel Ridge', 'SOAP_specific_small no CNAV'): [],
 ('Kernel Ridge', 'Canonical ACE no CNAV'): [],
 ('Kernel Ridge', 'Canonical BOP no CNAV'): [],
 ('Kernel Ridge', 'SOAP_canonicalW_small'): [],
 ('Kernel Ridge', 'SOAP_canonicalW_small no CNAV'): []}

In [26]:
cleanFCresults = {}

In [29]:
for (modelname, featurename), list_problem_index  in problematic_estimators:
    cleanFCresults[(modelname, featurename)] = []
    if len(list_problem_index) < 0:
        continue
    for 

SyntaxError: invalid syntax (1746470960.py, line 5)